# Deber — Semana 3: Carga de Datos, Series/DataFrames, loc/iloc y Análisis de Texto
## Análisis de Datos (TDSD353) | Período 2026-A
## ESFOT — Escuela Politécnica Nacional


**Nombre:** Alan Javier Reyes Casierra    
**Fecha:** 25/4/2026
<br/>
---

### Objetivo de este Taller
Aplicar las técnicas aprendidas en clase para:
1. Cargar datos desde archivos **JSON**, **TXT** (CSV) y **XML** usando Pandas.
2. Detectar y documentar **problemas de calidad de datos**.
3. Unificar múltiples DataFrames con `pd.concat()` y exportar a CSV.
4. Comprender y usar **Series** y **DataFrames**.
5. Acceder a datos con **`loc`** (por etiqueta) e **`iloc`** (por posición).
6. Descargar y procesar texto desde la web con `urllib`.

### Archivos de trabajo
| Archivo | Contenido | IDs |
|---------|-----------|-----|
| `productos.json` | 20 productos de librería | 1–20 |
| `productos.txt` | 20 productos de librería (formato CSV) | 21–40 |
| `productos.xml` | 20 productos de librería | 41–60 |
| `netflix.csv` | Catálogo de Netflix (~8800 títulos) | — |

> ⚠️ Los archivos de productos contienen **errores intencionales** (`"error"`, `"N/A"`, `null`, stock negativo, campos vacíos) para ilustrar problemas reales de calidad de datos.

---

---
## PARTE A: Pipeline de datos — Inventario de Papelería

Tenemos 3 archivos con datos del inventario de una Papelería:
- `productos.json` — 20 productos (IDs 1–20)
- `productos.txt` — 20 productos (IDs 21–40, separados por coma)
- `productos.xml` — 20 productos (IDs 41–60)

Columnas compartidas: `id`, `nombre`, `precio_compra`, `categoria`, `stock`, `proveedor`, `precio_venta_publico`

> ⚠️ Los datos contienen errores intencionales (`"error"`, `"N/A"`, `null`, stock negativo, campos vacíos) para ilustrar problemas reales de calidad de datos.

### A-1. Importar librerías

| Librería | Uso |
|----------|-----|
| `pandas` | Manipulación de datos tabulares |
| `urllib.request` | Descarga de archivos desde la web |
| `collections.Counter` | Conteo de frecuencias |
| `time` | Medición de tiempos |
| `re` | Expresiones regulares para limpieza de texto |

In [2]:
import pandas as pd
from urllib import request
from collections import Counter
import time
import re

### A-2. Cargar JSON con `pd.read_json()`

**JSON** (JavaScript Object Notation): formato ligero muy común en APIs web.

```python
df = pd.read_json('archivo.json')
```

Parámetros útiles: `orient` (formato del JSON), `encoding`, `dtype` (forzar tipos).

In [3]:
df_json = pd.read_json('productos.json')
df_json.head()

,id,nombre,precio_compra,categoria,stock,proveedor,precio_venta_publico
0,1,Bolígrafo BIC Azul,0.35,Escritura,500,DistribuPapel S.A.,0.75
1,2,Cuaderno Universitario 100 hojas,1.8,Cuadernos,300,Norma Ecuador,3.50
2,3,Lápiz HB Faber-Castell,0.2,Escritura,800,DistribuPapel S.A.,0.45
3,4,Resaltador Amarillo,error,Escritura,150,OfficeMax Ecuador,1.20
4,5,Carpeta Manila A4,0.15,Archivo,1000,PapelPro Cía. Ltda.,0.30


### A-3. Cargar TXT con `pd.read_csv()`

**TXT tabular** (datos separados por coma): los archivos `.txt` con formato de tabla se leen con `read_csv()` ajustando el separador.

```python
df = pd.read_csv('archivo.txt', sep=',')   # separador: coma
df = pd.read_csv('archivo.txt', sep='\t')  # separador: tabulación
```

> **Diferencia CSV vs TXT**: ambos son texto plano. La extensión `.txt` no implica un formato fijo — siempre revisa el separador antes de cargar.

In [4]:
df_txt = pd.read_csv('productos.txt')
df_txt.head()

,id,nombre,precio_compra,categoria,stock,proveedor,precio_venta_publico
0,21,Sacapuntas Doble Uso,0.15,Escritura,700,DistribuPapel S.A.,0.35
1,22,Cuaderno Espiral A5 80 hojas,1.40,Cuadernos,220,Norma Ecuador,2.80
2,23,Lápiz de Color x24,2.10,Manualidades,160,Norma Ecuador,4.20
3,24,Pegamento en Barra 40g,0.55,Manualidades,430,OfficeMax Ecuador,1.05
4,25,Papel Lustre x10 colores,0.60,Papel,380,PapelPro Cía. Ltda.,1.20


### A-4. Cargar XML con `pd.read_xml()`

**XML** (eXtensible Markup Language): formato jerárquico usado en servicios web SOAP y datos gubernamentales.

```python
df = pd.read_xml('archivo.xml', xpath='.//producto')
```

El parámetro `xpath` indica qué nodos representan cada fila. 

In [5]:
#%pip install lxml

df_xml = pd.read_xml('productos.xml', xpath='.//producto')
df_xml.head()

,id,nombre,precio_compra,categoria,stock,proveedor,precio_venta_publico
0,41,Tiza Blanca Caja x100,1.5,Escritura,180,Norma Ecuador,3.00
1,42,Papelógrafo Cuadriculado x50,2.8,Papel,130,PapelPro Cía. Ltda.,5.20
2,43,Organizador de Escritorio Plástico,3.6,Organización,65,OfficeMax Ecuador,6.80
3,44,Cuaderno Cosido 50 hojas,0.9,Cuadernos,texto,Norma Ecuador,1.80
4,45,Cartulina de Colores x10,0.7,Papel,320,PapelPro Cía. Ltda.,1.40


Al ejecutar la celda anterior puede existir un error: `ImportError: lxml not found, please install or use the etree parser.`  
Lo solucionamos con: `%pip install lxml`

### A-5. Verificar consistencia y detectar errores

Antes de concatenar, verificamos que los 3 DataFrames tengan las **mismas columnas** y documentamos los problemas de calidad.

Errores comunes en datos reales:
- Valores `null` / `NaN` (datos faltantes)
- Texto donde debería haber números (`"error"`, `"N/A"`, `"texto"`)
- Valores lógicamente inválidos (Ejemplo stock negativo)
- Campos vacíos (`""`)

In [6]:
print(f'Son iguales los 3 dataframes? {set(df_txt.columns) == set(df_json.columns) == set(df_xml.columns)}')

Son iguales los 3 dataframes? True


In [7]:
print('Valores vacíos por columna de cada df')
# print(f'df_xml\n{df_xml.isnull().sum()} \ndf_txt\n{df_txt.isnull().sum()} \ndf_json\n{df_json.isnull().sum()}')

dfs = {
    'json': df_json,
    'xml': df_xml,
    'txt': df_txt
}

for nombre, df in dfs.items():
    print(f'\t---{nombre.upper()}---\n{df.isnull().sum()}')

Valores vacíos por columna de cada df
	---JSON---
id                      0
nombre                  0
precio_compra           1
categoria               0
stock                   0
proveedor               1
precio_venta_publico    0
dtype: int64
	---XML---
id                      0
nombre                  0
precio_compra           2
categoria               0
stock                   0
proveedor               1
precio_venta_publico    0
dtype: int64
	---TXT---
id                      0
nombre                  0
precio_compra           2
categoria               0
stock                   0
proveedor               0
precio_venta_publico    0
dtype: int64


In [8]:
print(f'Valores inválidos de cada df')
for nombre, df in dfs.items():
    print(nombre.upper())
    df['stock'] = pd.to_numeric(df['stock'], errors='coerce')
    display(df[df['stock'] < 0])

Valores inválidos de cada df
JSON


,id,nombre,precio_compra,categoria,stock,proveedor,precio_venta_publico
19,20,Crayones x12 Colores,0.95,Manualidades,-30.0,Norma Ecuador,1.8


XML


,id,nombre,precio_compra,categoria,stock,proveedor,precio_venta_publico
16,57,Cartón Corrugado 50x70cm,0.8,Manualidades,-15.0,PapelPro Cía. Ltda.,1.60


TXT


,id,nombre,precio_compra,categoria,stock,proveedor,precio_venta_publico


In [9]:
for nombre, df in dfs.items():
    print(f'\t---{nombre.upper()}---')
    print((df == '').sum())

	---JSON---
id                      0
nombre                  0
precio_compra           0
categoria               0
stock                   0
proveedor               0
precio_venta_publico    0
dtype: int64
	---XML---
id                      0
nombre                  0
precio_compra           0
categoria               0
stock                   0
proveedor               0
precio_venta_publico    0
dtype: int64
	---TXT---
id                      0
nombre                  0
precio_compra           0
categoria               0
stock                   0
proveedor               0
precio_venta_publico    0
dtype: int64


### A-6. Concatenar con `pd.concat()` y exportar con el metodo to_csv()

`pd.concat()` apila DataFrames verticalmente. `ignore_index=True` reinicia el índice.

| Función | Uso |
|---------|-----|
| `pd.concat()` | Apilar DataFrames con **misma estructura** |

In [10]:
df_final = pd.concat([df_json, df_xml, df_txt], ignore_index=True)
df_final

,id,nombre,precio_compra,categoria,stock,proveedor,precio_venta_publico
0,1,Bolígrafo BIC Azul,0.35,Escritura,500.0,DistribuPapel S.A.,0.75
1,2,Cuaderno Universitario 100 hojas,1.8,Cuadernos,300.0,Norma Ecuador,3.5
2,3,Lápiz HB Faber-Castell,0.2,Escritura,800.0,DistribuPapel S.A.,0.45
3,4,Resaltador Amarillo,error,Escritura,150.0,OfficeMax Ecuador,1.2
4,5,Carpeta Manila A4,0.15,Archivo,1000.0,PapelPro Cía. Ltda.,0.3
5,6,Tijeras Escolares 13cm,0.9,Manualidades,200.0,OfficeMax Ecuador,1.8
6,7,Corrector Líquido 18ml,None,Escritura,180.0,Norma Ecuador,1.5
7,8,Goma de Borrar Blanca,0.25,Escritura,600.0,DistribuPapel S.A.,0.55
8,9,Resma Papel Bond A4 75g,3.2,Papel,250.0,PapelPro Cía. Ltda.,5.5
9,10,Marcador Permanente Negro,0.6,Escritura,400.0,OfficeMax Ecuador,1.1


Exportar el dataframe concatenado con `to_csv()` con index=False  como argumento adicional, evitar escribir el índice numérico como columna extra

In [11]:
df_final.to_csv('productos_final.csv', index=False)

---
---
## PARTE B: Series, DataFrames, `loc` e `iloc` — Catálogo de Netflix

Trabajaremos con el archivo `netflix.csv`, un dataset real con el catálogo de contenidos de Netflix.

| Columna | Descripción |
|---------|------------|
| `show_id` | ID único del título |
| `type` | `Movie` o `TV Show` |
| `title` | Nombre del título |
| `director` | Director(es) |
| `cast` | Elenco principal |
| `country` | País de producción |
| `date_added` | Fecha de añadido a Netflix |
| `release_year` | Año de estreno |
| `rating` | Clasificación de audiencia |
| `duration` | Duración (minutos o temporadas) |
| `listed_in` | Géneros/categorías |
| `description` | Sinopsis |

### B-1. Carga el dataset y realiza una primera inspección

In [12]:
netflix = pd.read_csv('netflix.csv')
print(f'Tamaño del dataset netflix: {netflix.shape}')
display(netflix.head())
netflix.info()

Tamaño del dataset netflix: (6137, 15)


,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.601,NaN
1,tm82169,Rocky,MOVIE,"When world heavyweight boxing champion, Apollo...",1976,PG,119,"['drama', 'sport']",['US'],NaN,tt0075148,8.1,588100.0,106.361,7.782
2,tm17823,Grease,MOVIE,Australian good girl Sandy and greaser Danny f...,1978,PG,110,"['romance', 'comedy']",['US'],NaN,tt0077631,7.2,283316.0,33.160,7.406
3,tm191099,The Sting,MOVIE,A novice con man teams up with an acknowledged...,1973,PG,129,"['crime', 'drama', 'comedy', 'music']",['US'],NaN,tt0070735,8.3,266738.0,24.616,8.020
4,tm69975,Rocky II,MOVIE,After Rocky goes the distance with champ Apoll...,1979,PG,119,"['drama', 'sport']",['US'],NaN,tt0079817,7.3,216307.0,75.699,7.246


<class 'pandas.DataFrame'>
RangeIndex: 6137 entries, 0 to 6136
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    6137 non-null   str    
 1   title                 6137 non-null   str    
 2   type                  6137 non-null   str    
 3   description           6114 non-null   str    
 4   release_year          6137 non-null   int64  
 5   age_certification     3394 non-null   str    
 6   runtime               6137 non-null   int64  
 7   genres                6137 non-null   str    
 8   production_countries  6137 non-null   str    
 9   seasons               2306 non-null   float64
 10  imdb_id               5741 non-null   str    
 11  imdb_score            5669 non-null   float64
 12  imdb_votes            5653 non-null   float64
 13  tmdb_popularity       6061 non-null   float64
 14  tmdb_score            5885 non-null   float64
dtypes: float64(5), int64(2), str(8)


---
### B-2. ¿Qué es una Series?

Una **Series** es un arreglo **unidimensional** etiquetado. Cada elemento tiene un **valor** y un **índice**.

```python
# Desde una lista
s = pd.Series([10, 20, 30])

# Desde un diccionario (claves = índice)
s = pd.Series({'a': 10, 'b': 20, 'c': 30})
```

| Atributo | Descripción |
|----------|------------|
| `.values` | Datos como array NumPy |
| `.index` | Etiquetas del índice |
| `.dtype` | Tipo de dato |
| `.name` | Nombre de la Series |
| `.shape` | Forma (número de elementos,) |

**Cuando accedes a UNA columna de un DataFrame, obtienes una Series:**

### Extraer una columna = una Series del csv de netflix

In [13]:
netflix['title']
title = netflix['title']
title
# title.index
# title.dtype
# title.name
# title.shape

0       Five Came Back: The Reference Films
1                                     Rocky
2                                    Grease
3                                 The Sting
4                                  Rocky II
                       ...                 
6132                          عبود في البيت
6133                                Sweetie
6134              Sommore: Queen Chandelier
6135                           All Na Vibes
6136                        Going to Heaven
Name: title, Length: 6137, dtype: str

### Operaciones comunes sobre Series

### Obten las estadísticas de la columna 'release_year', count, min, max, mean, median, std

In [14]:
netflix['release_year']
release_year = netflix['release_year']

print(f'release_year.count(): {release_year.count()}')
print(f'release_year.min(): {release_year.min()}')
print(f'release_year.max(): {release_year.max()}')
print(f'release_year.mean(): {round(release_year.mean(), 2)}')
print(f'release_year.median(): {release_year.median()}')
print(f'release_year.std(): {round(release_year.std(), 2)}')

release_year.count(): 6137
release_year.min(): 1945
release_year.max(): 2023
release_year.mean(): 2017.37
release_year.median(): 2019.0
release_year.std(): 6.6


### Obten el Top 10 países con más contenido producido

In [15]:
netflix['production_countries']
production_countries = netflix['production_countries']

production_countries.value_counts().head(10)

production_countries
['US']    1981
['IN']     633
['JP']     278
['KR']     259
['GB']     235
['ES']     177
[]         176
['FR']     121
['MX']     115
['BR']     104
Name: count, dtype: int64

### Obten el Top 10 de series/peliculas mas vistas

In [16]:
top_10 = netflix.sort_values(by='imdb_votes', ascending=False).head(10)

print(top_10[['title', 'imdb_votes']])

                                                 title  imdb_votes
201                                    The Dark Knight   2684317.0
87                                        Forrest Gump   2106826.0
186                                       Breaking Bad   1936461.0
195  The Lord of the Rings: The Fellowship of the Ring   1895545.0
209      The Lord of the Rings: The Return of the King   1865989.0
208              The Lord of the Rings: The Two Towers   1684864.0
917                                    Stranger Things   1220079.0
88                                      Reservoir Dogs   1031133.0
187                                   The Walking Dead   1013253.0
533                                   The Hunger Games    930840.0


---
### B-3. Acceso con `loc` (por etiqueta)

**`loc`** selecciona datos por **etiqueta** del índice y **nombre** de columna.

```python
df.loc[etiqueta_fila]                    # Una fila completa
df.loc[etiqueta_fila, 'columna']         # Un valor específico
df.loc[inicio:fin]                       # Rango de filas (INCLUSIVO)
df.loc[condicion]                        # Filtrado booleano
df.loc[condicion, ['col1', 'col2']]      # Filtrado + columnas
```

> Con `loc`, los rangos son **INCLUSIVOS** en ambos extremos: `loc[2:5]` incluye 2, 3, 4 **y 5**.

In [23]:
# Genera un ejemplo de Acceso a una fila completa por su etiqueta de índice
netflix.loc[20]
netflix.loc[20, 'type']
netflix.loc[10:25]
netflix.loc[release_year > 2015]
netflix.loc[release_year > 2015, ['id', 'title', 'release_year']]

,id,title,release_year
917,ts38796,Stranger Things,2016
918,ts42069,The Crown,2016
919,ts41766,The Good Place,2016
925,tm143617,La La Land,2016
927,ts36147,Lucifer,2016
...,...,...,...
6132,tm1303784,عبود في البيت,2023
6133,tm1260999,Sweetie,2023
6134,tm1310286,Sommore: Queen Chandelier,2023
6135,tm1072700,All Na Vibes,2023


In [34]:
# Filtrado con MÚLTIPLES condiciones
# Obten el listado de Películas de Estados Unidos producidas desde 2018
netflix.loc[
    (netflix['release_year'] >= 2018) &
    (netflix['production_countries'].str.contains('US', na=False))
]


,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
1422,ts79813,You,SHOW,"A dangerously charming, intensely obsessive yo...",2018,TV-MA,48,"['drama', 'romance', 'thriller', 'crime']",['US'],4.0,tt7335184,7.7,261831.0,341.151,8.118
1423,ts81927,New Amsterdam,SHOW,The new medical director breaks the rules to h...,2018,TV-14,43,['drama'],['US'],5.0,tt7817340,8.0,43464.0,163.879,8.462
1428,ts78801,Cobra Kai,SHOW,This Karate Kid sequel series picks up 30 year...,2018,TV-14,33,"['action', 'drama', 'comedy', 'sport']",['US'],6.0,tt7221388,8.5,190785.0,218.009,8.200
1431,ts81294,Manifest,SHOW,After landing from a turbulent but routine fli...,2018,TV-14,42,"['scifi', 'drama', 'thriller']",['US'],4.0,tt8421350,7.1,71203.0,176.155,7.733
1432,ts83970,All American,SHOW,When a rising high school football player from...,2018,TV-14,43,"['drama', 'sport']",['US'],5.0,tt11574984,7.6,9704.0,75.706,8.209
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6109,ts367818,Bling Empire: New York,SHOW,"Meet a fresh group of wealthy, sophisticated a...",2023,TV-MA,40,['reality'],['US'],1.0,tt22481904,5.2,376.0,6.621,NaN
6121,tm1312789,Here Love Lies,MOVIE,"In the pursuit of 'love and forever', single p...",2023,NaN,131,"['thriller', 'romance']","['NG', 'US']",NaN,tt15139698,5.2,65.0,22.722,5.000
6122,ts364674,Princess Power,SHOW,Princess friends from four different Fruitdoms...,2023,TV-Y,15,"['family', 'fantasy', 'animation', 'comedy']",['US'],1.0,tt22013036,7.0,60.0,6.319,8.500
6126,tm1291873,Andrew Santino: Cheeseburger,MOVIE,No topic is safe in this unfiltered stand-up s...,2023,NaN,58,"['comedy', 'documentation']",['US'],NaN,tt24640480,6.5,808.0,3.181,5.500


In [40]:
# Filtrado con isin() para múltiples valores
# Filtra todas las filas cuyo Contenido tenga como origen Latinoamérica
paises_latam = ['Mexico', 'Argentina', 'Colombia', 'Chile', 'Brazil', 'Peru']
paises_latam_corregido = ["['MX']", "['AR']", "['CO']", "['CL']", "['BR']", "['PE']"]
netflix.loc[netflix['production_countries'].isin(paises_latam_corregido)]


,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
63,tm126769,Waiting for the Hearse,MOVIE,"Mama Cora, who is almost eighty years old, has...",1985,NaN,87,['comedy'],['AR'],NaN,tt0089108,8.0,6714.0,7.593,8.180
127,ts28516,Okupas,SHOW,"During the year 2000, Ricardo, Pollo, Walter a...",2000,TV-MA,40,"['drama', 'crime']",['AR'],1.0,tt0289649,9.0,2630.0,7.290,8.635
136,tm41525,Herod's Law,MOVIE,"Mexico, 1949. The fable of a janitor turned Ma...",2000,R,123,"['comedy', 'crime', 'drama']",['MX'],NaN,tt0221344,7.8,6414.0,12.915,7.983
155,ts224786,Escalona,SHOW,"The improbable real life of Rafael Escalona, w...",1991,TV-MA,44,['drama'],['CO'],1.0,NaN,NaN,NaN,8.981,7.612
247,ts38488,Hidden Passion,SHOW,The Reyes-Elizondo's idyllic lives are shatter...,2003,TV-14,43,"['drama', 'romance']",['CO'],2.0,tt0387763,7.8,2632.0,164.924,7.644
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6102,ts321003,The Endless Night,SHOW,"A fire at Nightclub Kiss, which killed 242 peo...",2023,TV-MA,43,"['crime', 'drama', 'thriller']",['BR'],1.0,tt16427924,7.7,1124.0,13.943,8.200
6103,ts374551,Against the Ropes,SHOW,They say being a woman is all a business and m...,2023,TV-MA,40,"['drama', 'comedy', 'sport']",['MX'],1.0,tt25031242,6.5,207.0,29.503,9.083
6110,tm1301192,Whindersson Nunes: Preaching to the Choir,MOVIE,The new show by comedian Whindersson Nunes ent...,2023,NaN,67,['comedy'],['BR'],NaN,tt26440342,6.4,182.0,11.810,7.200
6114,tm1297263,All the Places,MOVIE,An estranged brother and sister reunite at the...,2023,NaN,97,"['comedy', 'drama']",['MX'],NaN,tt12616964,5.6,249.0,292.518,6.800


In [44]:
# str.contains(): filtrado por texto parcial en columna
# Títulos que contengan 'Love' en el nombre

netflix.loc[netflix['title'].str.contains('Love', case=False, na=False)]


,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
299,tm97072,Eat Pray Love,MOVIE,Liz Gilbert had everything a modern woman is s...,2010,PG-13,133,"['drama', 'romance']",['US'],NaN,tt0879870,5.8,100026.0,19.052,6.230
322,tm464888,Love Aaj Kal,MOVIE,When professional ambitions clash with persona...,2009,PG-13,141,"['drama', 'comedy', 'romance']",['IN'],NaN,tt1275863,6.8,16443.0,3.097,5.500
418,ts4866,Fated to Love You,SHOW,"Fated to Love You, also known as You're My Des...",2008,TV-G,67,"['drama', 'comedy', 'romance']",['TW'],1.0,tt1483930,7.4,510.0,16.052,6.833
441,tm95434,Love in a Puff,MOVIE,When the Hong Kong government enacts a ban on ...,2010,NaN,104,"['comedy', 'drama', 'romance']",['HK'],NaN,tt1602479,7.1,2746.0,4.220,6.953
447,tm76399,A Love Story,MOVIE,Ian Montes is a picture of success. Despite be...,2007,NaN,117,"['romance', 'drama']",['PH'],NaN,tt0990433,6.5,150.0,1.410,4.600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6076,ts360024,In Love All Over Again,SHOW,"Year 2003, Irene a young film student who is p...",2023,TV-MA,45,"['drama', 'romance', 'comedy']",['ES'],1.0,tt21206964,6.3,669.0,85.435,6.400
6078,tm1304302,Love at First Kiss,MOVIE,"The story of Javier who, at the age of 16, whi...",2023,NaN,96,"['comedy', 'romance']",['ES'],NaN,tt14463514,5.7,897.0,152.671,6.246
6100,tm1300541,Squared Love All Over Again,MOVIE,The relationship of a well-known journalist an...,2023,NaN,99,"['comedy', 'romance']",['PL'],NaN,tt26369131,4.5,383.0,25.780,5.600
6101,ts314687,Love to Hate You,SHOW,"For a woman who despises losing to men, and a ...",2023,TV-MA,54,"['comedy', 'drama', 'romance']",['KR'],1.0,tt26229508,7.9,1584.0,36.117,8.300


### B-4. Ya que tienes acceso a los datos de Netflix, en base a tu curiosidad, usando pandas obten al menos 3 datos de interes adicionales, que te gustaria conocer.

In [47]:
print('type MOVIE')
netflix.loc[netflix['type'].str.contains('MOVIE'), ['title', 'release_year', 'genres']]

type MOVIE


,title,release_year,genres
1,Rocky,1976,"['drama', 'sport']"
2,Grease,1978,"['romance', 'comedy']"
3,The Sting,1973,"['crime', 'drama', 'comedy', 'music']"
4,Rocky II,1979,"['drama', 'sport']"
5,Monty Python and the Holy Grail,1975,"['fantasy', 'comedy']"
...,...,...,...
6132,عبود في البيت,2023,"['family', 'comedy']"
6133,Sweetie,2023,['documentation']
6134,Sommore: Queen Chandelier,2023,['comedy']
6135,All Na Vibes,2023,['drama']


In [60]:
print('Película con la mayor duración')
netflix.loc[[netflix['runtime'].argmax()]]

Película con la mayor duración


,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
486,tm25842,A Lion in the House,MOVIE,Five families struggle with the ups and downs ...,2006,NaN,225,['documentation'],['US'],NaN,tt0492472,8.7,338.0,2.821,6.5


In [68]:
print('Películas más actuales')
netflix.loc[netflix['release_year'] == 2023]

Películas más actuales


,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
6040,ts278785,Lockwood & Co,SHOW,"A tiny startup, run by two teenage boys and a ...",2023,TV-14,44,"['horror', 'thriller', 'action', 'drama', 'fan...",['GB'],1.0,tt13802576,7.4,15929.0,33.360,7.933
6041,ts378895,Murdaugh Murders: A Southern Scandal,SHOW,Shocking tragedies shatter a tight-knit South ...,2023,TV-MA,47,"['crime', 'documentation']",['US'],1.0,tt26395768,6.8,2779.0,11.443,6.500
6042,tm1260493,Your Place or Mine,MOVIE,When best friends and total opposites Debbie a...,2023,PG-13,99,"['romance', 'comedy']",['US'],NaN,tt12823454,5.6,26216.0,268.001,6.400
6043,ts327671,Full Swing,SHOW,Follows behind the scenes of what it takes to ...,2023,TV-MA,45,"['documentation', 'sport']",['US'],2.0,tt17050700,8.2,1763.0,6.485,6.000
6044,ts328355,Kaleidoscope,SHOW,A master thief and his crew attempt an epic an...,2023,TV-MA,42,"['crime', 'drama', 'action', 'thriller']",['US'],1.0,tt15438246,6.7,35159.0,112.966,7.148
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6132,tm1303784,عبود في البيت,MOVIE,Two young boys must work together to stop robb...,2023,NaN,81,"['family', 'comedy']",['KW'],NaN,NaN,NaN,NaN,3.351,2.000
6133,tm1260999,Sweetie,MOVIE,"‘Theatre is my life,’ Yıldız Kenter admits in ...",2023,NaN,120,['documentation'],['TR'],NaN,tt26349328,7.9,209.0,2.450,7.200
6134,tm1310286,Sommore: Queen Chandelier,MOVIE,This Queen of Comedy shines as she takes the s...,2023,NaN,69,['comedy'],['US'],NaN,tt21033382,6.1,91.0,1.960,NaN
6135,tm1072700,All Na Vibes,MOVIE,The lives of three teenagers and a hit-man int...,2023,NaN,80,['drama'],['NG'],NaN,tt14922926,5.2,18.0,1.357,4.000


---
### B-5. Acceso con `iloc` (por posición entera)

**`iloc`** selecciona datos por **posición numérica** (como índices de listas en Python).

```python
df.iloc[posicion]                          # Una fila
df.iloc[pos_fila, pos_columna]             # Un valor
df.iloc[inicio:fin]                        # Rango (EXCLUSIVO al final)
df.iloc[[0, 5, 10]]                        # Posiciones específicas
df.iloc[::n]                               # Cada n filas
```

> Con `iloc`, el rango es **EXCLUSIVO** al final: `iloc[2:5]` incluye posiciones 2, 3, 4 pero **NO 5**.

### Diferencia clave `loc` vs `iloc`:

| | `loc` | `iloc` |
|--|-------|--------|
| **Acceso** | Por **etiqueta** | Por **posición** |
| **Rango** | **Inclusivo** | **Exclusivo** |
| **Columnas** | Por nombre | Por número (0, 1, 2...) |
| **Filtrado** |  Booleano | No directo |

In [84]:
# Obten la Primera fila (posición 0) usando loc
netflix.iloc[[0], :]

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.601,NaN


In [86]:
# Última fila (índice negativo, como listas de Python) usando loc
netflix.loc[[-1], :]

KeyError: 'None of [RangeIndex(start=-1, stop=0, step=1)] are in the [index]'

Si queremos mostrar la última fila debemos usar `iloc`, que puede manejar -1 para ubicarse, loc no

In [87]:
netflix.iloc[[-1], :]

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
6136,tm561709,Going to Heaven,MOVIE,"A story about young boy sultan, 11 years old l...",2023,NaN,90,['family'],['AE'],NaN,tt4623222,7.0,40.0,NaN,NaN


al igual que antes debemos usar iloc para evitar errores porque estamos accediendo por posiciones numéricas

In [95]:
# Valor específico: fila 10, columna 2 (title) usando loc
netflix.iloc[[10], [1]]

,title
10,Heroes


In [97]:
# Rango de filas (EXCLUSIVO: posiciones 2, 3, 4 — NO incluye 5) usando loc
netflix.iloc[2:5]

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
2,tm17823,Grease,MOVIE,Australian good girl Sandy and greaser Danny f...,1978,PG,110,"['romance', 'comedy']",['US'],NaN,tt0077631,7.2,283316.0,33.160,7.406
3,tm191099,The Sting,MOVIE,A novice con man teams up with an acknowledged...,1973,PG,129,"['crime', 'drama', 'comedy', 'music']",['US'],NaN,tt0070735,8.3,266738.0,24.616,8.020
4,tm69975,Rocky II,MOVIE,After Rocky goes the distance with champ Apoll...,1979,PG,119,"['drama', 'sport']",['US'],NaN,tt0079817,7.3,216307.0,75.699,7.246


In [103]:
# Usando loc , obten los 10 títulos más antiguos en Netflix
# Combinamos sort_values con iloc(Ejemplo base: netflix_ordenado = df_netflix.sort_values("release_year", ascending=True))
print('=== 10 títulos más antiguos en Netflix ===')
netflix.sort_values('release_year', ascending=True).head(10)

=== 10 títulos más antiguos en Netflix ===


,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.601,NaN
27,tm19608,The Blazing Sun,MOVIE,A rich landlord floods and destroys a village ...,1954,NaN,100,"['drama', 'crime', 'romance']",['EG'],NaN,tt0047500,7.4,1219.0,2.854,6.865
9,tm16479,White Christmas,MOVIE,Two talented song-and-dance men team up after ...,1954,NaN,120,"['comedy', 'music', 'romance']",['US'],NaN,tt0047673,7.5,46586.0,11.598,7.200
26,tm204541,Dark Waters,MOVIE,"Ragab, a poor sailor, returns home to Alexandr...",1956,NaN,120,"['drama', 'romance', 'thriller']",['EG'],NaN,tt0049761,6.7,703.0,2.999,5.556
12,tm135083,Cairo Station,MOVIE,"Qinawi, a physically challenged peddler who ma...",1958,NaN,77,"['drama', 'comedy', 'crime']",['EG'],NaN,tt0051390,7.5,4878.0,7.372,7.400
22,tm10204,Professor,MOVIE,Sita devi is a very strict aunt for a number o...,1962,NaN,163,"['drama', 'comedy', 'romance']",['IN'],NaN,tt0056379,6.9,312.0,1.911,5.800
24,tm27298,Saladin the Victorious,MOVIE,The first Sultan of Egypt and Syria leads the ...,1963,NaN,186,"['drama', 'history', 'romance', 'war', 'action']",['EG'],NaN,tt0057357,7.6,2670.0,6.781,7.100
19,tm71909,Amrapali,MOVIE,"After a failed conquest, Emperor Ajaatshatru p...",1966,NaN,120,['fantasy'],['IN'],NaN,tt0060104,6.7,251.0,2.805,6.300
15,tm75975,Prince,MOVIE,"To better himself, a spoiled prince temporaril...",1969,NaN,152,['romance'],['IN'],NaN,tt0064842,6.8,199.0,2.450,7.000
7,ts22164,Monty Python's Flying Circus,SHOW,A British sketch comedy series with the shows ...,1969,TV-14,30,"['comedy', 'european']",['GB'],4.0,tt0063929,8.8,75654.0,24.773,8.258


---
---
## PARTE C: Procesamiento de Texto — *Dracula* (Bram Stoker)

En esta sección crear las cajas pertinentes a fin de realizar las siguientes actividades sobre el libro de Drácula:
1. Descargar texto desde una URL con `urllib`, crear una variable time, que contabilize el tiempo desde que se inicia la descarga.
2. Eliminar encabezados y pies de página propios de Project Gutenberg.
3. Eliminar signos de puntuación, números y caracteres especiales con `re`.
4. Procesar texto: minúsculas, tokenizar, filtrar.
5. Contar frecuencias con `collections.Counter`.
6. Mostrar el top 20 de las palabras mas frecuentes.
7. Guardar resultados en un archivo de texto(txt), donde se muestre el top 20 de palabras mas frecuentes y su cantidad de incidencias, asi como el tiempo total demorado desde que se descarga el libro, hasta que se cuenta las palabras mas frecuentes.

Usaremos **"Dracula"** de Bram Stoker, disponible en [Project Gutenberg](https://www.gutenberg.org/ebooks/345).  
https://www.gutenberg.org/cache/epub/345/pg345.txt



In [112]:
#Desarrollo del estudiante

# 1
url = 'https://www.gutenberg.org/cache/epub/345/pg345.txt'

inicio = time.time()
respuesta = request.urlopen(url)
texto = respuesta.read().decode('utf-8')

fin = time.time()
tiempo_descarga_ms = (fin - inicio) * 1000

# 2
posicion_inicio = texto.find('*** START OF THE PROJECT GUTENBERG EBOOK DRACULA ***')
posicion_fin = texto.find('*** END OF THE PROJECT GUTENBERG EBOOK DRACULA ***')

texto_limpio = texto[posicion_inicio:posicion_fin]

# 3
texto_solo_letras = re.sub(r'[^a-zA-z\s]', ' ', texto_limpio)

# 4
texto_minusculas = texto_limpio.lower()
palabras = texto_minusculas.split()
palabras_filtradas = [p for p in palabras if len(p) > 1]

# 5
contador = Counter(palabras_filtradas)

# 6
top_20  = contador.most_common(20)
print('TOP 20 palabras más frecuentes')
for palabra, frecuencia in top_20:
    print(f'{palabra} ---> {frecuencia}')

# 7
with open('resultado_dracula.txt', 'w', encoding='utf-8') as f:
    f.write('TOP 20 palabras más frecuentes\n\n')
    
    for palabra, frecuencia in top_20:
        f.write(f'{palabra} ---> {frecuencia}\n')

    f.write('\n')
    f.write(f'Tiempo de descarga (ms): {round(tiempo_descarga_ms, 2)}\n')


TOP 20 palabras más frecuentes
the ---> 7811
and ---> 5686
to ---> 4427
of ---> 3595
he ---> 2507
in ---> 2416
that ---> 2349
was ---> 1801
it ---> 1723
as ---> 1553
we ---> 1484
for ---> 1463
his ---> 1445
is ---> 1432
not ---> 1291
with ---> 1260
my ---> 1212
you ---> 1129
at ---> 1066
have ---> 1044
